# Day 7 Capstone Walkthrough — Multi-Model AI Assistant

---

This notebook drives the FastAPI capstone end-to-end using `httpx`.

**Start the server first**, in another terminal:
```bash
uvicorn main:app --reload
```

Then run through the cells below.

In [ ]:
!pip install httpx --quiet

In [ ]:
import httpx, json
BASE = "http://127.0.0.1:8000"
print(httpx.get(f"{BASE}/health").json())

## 1. Log in and grab a JWT

Demo users are hardcoded: `alice / wonderland` and `bob / builder`.

In [ ]:
r = httpx.post(f"{BASE}/auth/token", json={"username":"alice", "password":"wonderland"})
r.raise_for_status()
token = r.json()["access_token"]
H = {"Authorization": f"Bearer {token}"}
print("token[:40]:", token[:40], "...")

## 2. Simple chat — routes to Together AI (default cheap tier)

In [ ]:
r = httpx.post(f"{BASE}/chat", headers=H, json={"prompt": "Say hi in 5 words."}, timeout=60)
print(json.dumps(r.json(), indent=2))

## 3. Code prompt — routes to Claude (if key set)

Contains `python` and `refactor` → the router picks Claude Sonnet.

In [ ]:
code_prompt = "Refactor this Python code to use a list comprehension:\n" \
              "def double(xs):\n    out = []\n    for x in xs:\n        out.append(x*2)\n    return out"
r = httpx.post(f"{BASE}/chat", headers=H, json={"prompt": code_prompt}, timeout=60)
out = r.json()
print("provider :", out["provider"])
print("model    :", out["model"])
print("reason   :", out["reason"])
print("cost_usd :", out["cost_usd"])
print("---")
print(out["text"])

## 4. Streaming chat

Tokens appear as they're generated. The `X-Route-*` headers tell you which provider was picked.

In [ ]:
with httpx.stream(
    "POST", f"{BASE}/chat/stream", headers=H,
    json={"prompt": "Write a 3-line haiku about caching."},
    timeout=60,
) as r:
    print("provider :", r.headers.get("X-Route-Provider"))
    print("model    :", r.headers.get("X-Route-Model"))
    print("reason   :", r.headers.get("X-Route-Reason"))
    print("---")
    for chunk in r.iter_text():
        print(chunk, end="", flush=True)
print()

## 5. Force a specific model

Pass `force_model="provider/model"` to skip the router (great for A/B tests).

In [ ]:
r = httpx.post(f"{BASE}/chat", headers=H,
               json={"prompt":"Say hi.", "force_model":"together/mistralai/Mistral-7B-Instruct-v0.3"},
               timeout=60)
print(json.dumps(r.json(), indent=2))

## 6. Privacy mode → Together AI

When `privacy=True` the router always picks an open-source model on Together AI. In production you could swap this for a self-hosted deployment.

In [ ]:
r = httpx.post(f"{BASE}/chat", headers=H,
               json={"prompt":"Summarize: The Roman Empire was...", "privacy": True},
               timeout=60)
print(json.dumps(r.json(), indent=2))

## 7. Structured extraction

`/extract` runs Together AI in JSON mode and returns a typed `PersonExtract(name, age, email)`.

In [ ]:
r = httpx.post(f"{BASE}/extract", headers=H,
               json={"text": "Rohan Mehta is 34 years old and can be reached at rohan.mehta@example.com."},
               timeout=60)
print(r.json())

## 8. Usage report

Totals for the last 24 hours — this is what a billing dashboard would query.

In [ ]:
print(json.dumps(httpx.get(f"{BASE}/usage/me", headers=H).json(), indent=2))

## 9. Budget cap (optional)

Set `BUDGET_USD_PER_DAY=0.0001` in your env before launching uvicorn, then hit `/chat` twice. You'll get a `429` once cumulative cost exceeds the budget.

---

🎉 **Section 4 complete.** You built:
- Multi-provider routing (Together AI + OpenAI + Claude)
- Blocking + streaming chat endpoints
- Structured JSON extraction
- Cost tracking + budget enforcement

Section 5 will layer embeddings and semantic search on top.